In [1]:
!pip install py-vncorenlp vncorenlp rouge-score sacrebleu transformers gdown

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 49.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 90.0 MB/s eta 0:00:00
  Created wheel for py-vncorenlp: filename=py_vncorenlp-0.1.4-py3-none-any.whl size=4304 sha256=53d7e4f13cdea2da1ea79ed67dcfe8e40198e6c5cd098db62df9baf931fd60de
  Stored in directory: /root/.cache/pip/wheels/db/e5/ff/f4a1b4ece36e8582db1ca71150a34e987e65df50c35974e9bb
  Created wheel for vncorenlp: filename=vncorenlp-1.0.3-py3-none-any.whl size=2645933 sha256=baf29de236ce915cd6e1328b4c2f36cdd948774c761d4c6b6e56084b8632b039
  Stored in directory: /root/.cache/pip/wheels/6f/19/20/ec7083125fd06db1a19d0d3ca18806ecf4e8ed1464713b4efa
  Created wheel for rouge-s

# Config

In [2]:
import re
import math
import random

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from transformers import AutoTokenizer, AutoModel

import gdown
import sacrebleu

import py_vncorenlp
py_vncorenlp.download_model(save_dir='./')

from vncorenlp import VnCoreNLP
from rouge_score import rouge_scorer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "vinai/phobert-base"
MAX_LEN = 128
SEED = 42

print("Device:", DEVICE)


Device: cuda


# Utils

In [3]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.strip()
    if not text:
        return ""
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text


In [4]:
rdr = VnCoreNLP("./VnCoreNLP-1.2.jar", annotators='wseg', max_heap_size='-Xmx2g')

scorer = rouge_scorer.RougeScorer(
    ['rouge1', 'rouge2', 'rougeL'],
    use_stemmer=False
)


In [5]:
def split_sentences(text):
    if not isinstance(text, str):
        return []
    sents = rdr.tokenize(text)
    sents = [" ".join(sent) for sent in sents if sent]
    sents = [re.sub(r"_", " ", sent) for sent in sents]
    return sents

def score_sentences_in_doc(sentences, model, tokenizer, max_len=128):
    """
    Trả về list xác suất (0–1) cho từng câu.
    """
    model.eval()
    device = next(model.parameters()).device

    all_probs = []
    for s in sentences:
        enc = tokenizer(
            s,
            padding="max_length",
            truncation=True,
            max_length=max_len,
            return_tensors="pt",
        )
        enc = {k: v.to(device) for k, v in enc.items()}

        with torch.no_grad():
            _, logits = model(
                input_ids=enc["input_ids"],
                attention_mask=enc["attention_mask"],
                labels=None,
            )
            prob = torch.sigmoid(logits).item()
        all_probs.append(prob)

    return all_probs


In [6]:
def build_extractive_summary_ratio(
    doc_text,
    model,
    tokenizer,
    keep_ratio=0.3,
    max_len=128,
    min_sentences=1,
    max_sentences=None,
):
    """
    Tóm tắt extractive theo % số câu:
      - keep_ratio: tỷ lệ câu giữ lại, ví dụ 0.3 = 30%
      - min_sentences: tối thiểu số câu giữ lại
      - max_sentences: tối đa số câu giữ lại (None = không giới hạn)
    """
    sents = split_sentences(doc_text)
    if not sents:
        return ""

    n = len(sents)
    k = max(min_sentences, int(math.ceil(n * keep_ratio)))
    if max_sentences is not None:
        k = min(k, max_sentences)
    k = min(k, n)

    probs = score_sentences_in_doc(sents, model, tokenizer, max_len=max_len)

    idx = np.argsort(probs)[::-1][:k]
    idx_sorted = sorted(idx)
    selected_sents = [sents[i] for i in idx_sorted]
    return " ".join(selected_sents)

def build_extractive_summary_topk(
    doc_text,
    model,
    tokenizer,
    top_k=3,
    max_len=128,
):
    """
    Tóm tắt extractive theo số câu cố định:
      - top_k: số câu muốn giữ lại (ví dụ 3 câu quan trọng nhất)
    """
    sents = split_sentences(doc_text)
    if not sents:
        return ""

    probs = score_sentences_in_doc(sents, model, tokenizer, max_len=max_len)

    top_k = min(top_k, len(sents))
    idx = np.argsort(probs)[::-1][:top_k]
    idx_sorted = sorted(idx)
    selected_sents = [sents[i] for i in idx_sorted]
    return " ".join(selected_sents)


# Model

In [7]:
class PhoBERTClassifier(nn.Module):
    """
    PhoBERT + linear head nhị phân.
    Dùng CLS embedding (vị trí 0).
    """
    def __init__(self, model_name: str, dropout: float = 0.1):
        super().__init__()
        self.phobert = AutoModel.from_pretrained(model_name)
        hidden_size = self.phobert.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, 1)

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.phobert(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        cls_emb = outputs.last_hidden_state[:, 0, :]  # (batch, hidden)
        x = self.dropout(cls_emb)
        logits = self.classifier(x).squeeze(-1)       # (batch,)

        loss = None
        if labels is not None:
            loss_fct = nn.BCEWithLogitsLoss()
            loss = loss_fct(logits, labels)

        return loss, logits


# Load weight

In [8]:
weight_id = "1ph9GmQ0d_Zy79J8A3Ha6-z6zBf-9id0-"
weight_url = f"https://drive.google.com/uc?id={weight_id}"
best_ckpt_path = "best_phobert_extractive.pt"

gdown.download(weight_url, best_ckpt_path, quiet=False)


Downloading...
From (original): https://drive.google.com/uc?id=1ph9GmQ0d_Zy79J8A3Ha6-z6zBf-9id0-
From (redirected): https://drive.google.com/uc?id=1ph9GmQ0d_Zy79J8A3Ha6-z6zBf-9id0-&confirm=t&uuid=049308f8-35fe-4e2e-a923-138505740b0d
To: /content/best_phobert_extractive.pt
100%|██████████| 540M/540M [00:08<00:00, 66.3MB/s]


'best_phobert_extractive.pt'

# Load model

In [9]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

model = PhoBERTClassifier(MODEL_NAME)
state_dict = torch.load(best_ckpt_path, map_location=DEVICE)
model.load_state_dict(state_dict)
model.to(DEVICE)
model.eval()

print("Model + weight loaded xong.")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Model + weight loaded xong.


# Inference

In [10]:
def summarize(doc_text, keep_ratio=0.3, top_k=None):
    """
    Nhận 1 văn bản tiếng Việt, trả về tóm tắt extractive.
    keep_ratio: % số câu giữ lại, mặc định 30%.
    """
    doc_text = clean_text(doc_text)
    if not doc_text:
        return ""

    if top_k is not None:
      summary = build_extractive_summary_topk(
          doc_text,
          model,
          tokenizer,
          top_k=top_k
      )
    else:
      summary = build_extractive_summary_ratio(
          doc_text,
          model,
          tokenizer,
          keep_ratio=keep_ratio,
          max_len=MAX_LEN,
          min_sentences=1,
          max_sentences=None,
      )
    return summary


In [13]:
sample_text = """
Hehehehe. Hohooho
"""

print("=== TÓM TẮT ===")
# Tóm tắt bằng số câu
print(summarize(sample_text, top_k=2))
# Tóm tắt bằng % câu giữ lại
print(summarize(sample_text, keep_ratio=0.3))


=== TÓM TẮT ===
Hehehehe . Hohooho
Hohooho
